In [ ]:
def clique(
    X,
    y,
    model_builder="rf",
    folds=5,
    nsim=26,
    quantile_grid=True,
    keras_model=False,
    epochs=50,
    batch_size=64,
    random_state=123
):
    """
    Fast regression version of CLIQUE with:
      - NumPy-backed computation
      - Stored CV splits
      - Vectorized prediction calls
      - Batched Keras prediction
      - Same return structure as original

    Returns
    -------
    dict:
        {
            "models": list of fitted models,
            "local_imp": pd.DataFrame
        }
    """

    # ============================================================
    # Convert once
    # ============================================================

    X_df = pd.DataFrame(X)
    y_ser = pd.Series(y)

    feature_names = X_df.columns.tolist()

    X_np = X_df.to_numpy(dtype=np.float32)
    y_np = y_ser.to_numpy(dtype=np.float32)

    n, p = X_np.shape

    # ============================================================
    # CV splits (stored once)
    # ============================================================

    cv = KFold(
        n_splits=folds,
        shuffle=True,
        random_state=random_state
    )

    splits = list(cv.split(X_np))

    # ============================================================
    # Train fold models
    # ============================================================

    models = []

    for train_idx, test_idx in splits:

        # ----------------------------------------
        # Build model
        # ----------------------------------------

        if model_builder == "rf":

            model = RandomForestRegressor(
                n_estimators=100,
                random_state=random_state,
                n_jobs=-1
            )

        elif keras_model:

            model = model_builder(p)

        else:

            model = model_builder

        # ----------------------------------------
        # Fit model
        # ----------------------------------------

        if keras_model:

            model.fit(
                X_np[train_idx],
                y_np[train_idx],
                validation_data=(
                    X_np[test_idx],
                    y_np[test_idx]
                ),
                epochs=epochs,
                batch_size=batch_size,
                verbose=0
            )

        else:

            model.fit(
                X_np[train_idx],
                y_np[train_idx]
            )

        models.append(model)

    # ============================================================
    # Base OOF predictions
    # ============================================================

    preds = np.zeros(n, dtype=np.float32)

    for fold, (_, test_idx) in enumerate(splits):

        model = models[fold]

        if keras_model:

            preds[test_idx] = (
                model.predict(
                    X_np[test_idx],
                    batch_size=1024,
                    verbose=0
                ).ravel()
            )

        else:

            preds[test_idx] = model.predict(
                X_np[test_idx]
            )

    # MAE baseline
    base_loss = np.abs(y_np - preds)

    # ============================================================
    # Local importance matrix
    # ============================================================

    local_imp = np.zeros((n, p), dtype=np.float32)

    # ============================================================
    # Main importance loop
    # ============================================================

    for j in range(p):

        col_vals = X_np[:, j]

        # ----------------------------------------
        # Grid
        # ----------------------------------------

        if quantile_grid:

            grid_vals = np.quantile(
                col_vals,
                np.linspace(0, 1, nsim)
            )

        else:

            grid_vals = np.linspace(
                col_vals.min(),
                col_vals.max(),
                nsim
            )

        grid_vals = np.unique(grid_vals)

        # accumulator
        imp_accum = np.zeros(n, dtype=np.float32)

        # ====================================================
        # Fold-specific predictions
        # ====================================================

        for fold, (_, test_idx) in enumerate(splits):

            model = models[fold]

            X_test = X_np[test_idx]

            n_test = len(test_idx)
            n_grid = len(grid_vals)

            # ------------------------------------------------
            # Create giant stacked matrix
            # shape:
            #   (n_grid * n_test, p)
            # ------------------------------------------------

            X_big = np.repeat(
                X_test,
                repeats=n_grid,
                axis=0
            )

            # ------------------------------------------------
            # Replace column j with grid values
            # ------------------------------------------------

            X_big[:, j] = np.tile(grid_vals, n_test)

            # ------------------------------------------------
            # ONE prediction call
            # ------------------------------------------------

            if keras_model:

                pred_big = model.predict(
                    X_big,
                    batch_size=4096,
                    verbose=0
                ).ravel()

            else:

                pred_big = model.predict(X_big)

            # ------------------------------------------------
            # Reshape:
            #   rows = grid vals
            #   cols = observations
            # ------------------------------------------------

            pred_big = pred_big.reshape(n_test, n_grid).T

            # ------------------------------------------------
            # Compute MAE changes
            # ------------------------------------------------

            new_loss = np.abs(
                y_np[test_idx][None, :] - pred_big
            )

            delta = (
                new_loss - base_loss[test_idx][None, :]
            ).mean(axis=0)

            imp_accum[test_idx] = delta

        local_imp[:, j] = imp_accum

        print(f"Finished variable {j+1}/{p}")

    # ============================================================
    # Return same structure as original
    # ============================================================

    local_imp_df = pd.DataFrame(
        local_imp,
        columns=feature_names
    )

    return {
        "models": models,
        "local_imp": local_imp_df
    }

In [ ]:
# Code for Data

In [ ]:
res = clique(X=pd.DataFrame(X), y=pd.Series(y), model_builder="rf", nsim=25, folds=5)

In [ ]:
# SHAP
import shap

mod # = insert model code here
explainer_shap = shap.Explainer(model.predict, pd.DataFrame(X).values)
shap_values = explainer_shap(pd.DataFrame(X).values)

# extract the shap values for the first variable
shap_v1 = shap_values.values[:, 0]